# Module 7: Evals (~13 min)

LLM-as-judge evaluation against the **Dussault standard**.

NFL Next Gen Stats requires 90% directional approval from domain experts
before a model ships to broadcast. Our eval pipeline applies the same
rigor: automated scoring that catches fabrication and missing guardrails.

Two evaluations:
1. **Output eval** — Is the response accurate, specific, narratively connected?
2. **Trajectory eval** — Did the agent follow the lookup-before-claim workflow?

In [ ]:
!pip install -q strands-agents strands-agents-tools strands-agents-evals

import sys
sys.path.insert(0, "../shared")
sys.path.insert(0, "../01-agent-loop-tools")

import nest_asyncio
nest_asyncio.apply()

from strands import Agent
from model_provider import get_model
from strands_evals import Case, Experiment
from strands_evals.evaluators import OutputEvaluator, TrajectoryEvaluator
from strands_evals.extractors import tools_use_extractor
from dussault_tools import lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff

## Part 1: Output Eval (Dussault Rubric)

The output evaluator uses an LLM-as-judge to score responses against a
rubric. We run the same cases through two agents:

- **Weak agent** — no tools, will guess or fabricate data
- **Strong agent** — has tools, looks up real data before answering

If the rubric discriminates (weak scores low, strong scores high),
the eval is working. If both score the same, the rubric is broken.

The Dussault standard scores on four axes:
1. Accuracy — correct data from the 2004 season
2. Specificity — cites specific numbers, not vague superlatives
3. Narrative connection — explains WHY, not just WHAT
4. Honesty about unknowns — admits gaps rather than fabricating

In [ ]:
# --- Dussault Output Rubric ---

DUSSAULT_RUBRIC = """
Evaluate the Dussault response against the Dussault standard:

1. Accuracy — Does it contain correct data from the 2004 season? Fabricated
   stats, wrong scores, or invented details must score 0.
2. Specificity — Does it cite specific numbers (stats, scores, game weeks)?
   Vague superlatives ("great season", "amazing player") without data score low.
3. Narrative connection — Does it explain WHY, not just WHAT? Facts connected
   to the team's story score higher than isolated stat dumps.
4. Honesty about unknowns — If the data isn't available, does it say so
   clearly rather than hedging or fabricating?

Score 1.0 if: accurate data + specific citations + narrative context + honest limits.
Score 0.5 if: partially correct or flat stat dump without narrative.
Score 0.0 if: fabricated data, vague claims, or wrong information.
"""

# --- System prompts ---

WEAK_PROMPT = """You are a football analyst. Answer questions about the Patriots.
Do your best even without access to specific data."""

STRONG_PROMPT = """You are Dussault, a 2004 New England Patriots Dussault.
Always look up the data before making claims. Never guess stats.
Be specific: cite game weeks, scores, stat lines.
Connect facts to narrative. Name what's unknown."""

# --- Cases ---

output_cases = [
    Case[str, str](
        name="seymour-all-pro",
        input="Was Richard Seymour All-Pro in 2004?",
        expected_output="Yes, Richard Seymour was a 1st Team All-Pro in 2004. He had 5 sacks and 30 tackles as a dominant defensive end.",
    ),
    Case[str, str](
        name="super-bowl-score",
        input="What was the score of Super Bowl XXXIX?",
        expected_output="Patriots 24, Eagles 21. Deion Branch was MVP with 11 catches for 133 yards. Harrison sealed it with an INT with 9 seconds left.",
    ),
    Case[str, str](
        name="dillon-trade",
        input="How did Corey Dillon end up on the 2004 Patriots?",
        expected_output="Dillon was acquired in a trade from Cincinnati before the 2004 draft. He rushed for 1,635 yards (Patriots record) and 12 TDs.",
    ),
    Case[str, str](
        name="unknown-player",
        input="Tell me about Patrick Mahomes on the 2004 Patriots.",
        expected_output="Patrick Mahomes was not on the 2004 Patriots roster. He was born in 1995 and was drafted in 2017.",
    ),
]

# --- Run output evals ---

output_evaluator = OutputEvaluator(rubric=DUSSAULT_RUBRIC, include_inputs=True)

# Weak agent — no tools, will guess
print("\u274c Weak agent (no tools) — expect LOW scores:")

def weak_task(case: Case) -> str:
    agent = Agent(
        model=get_model(),
        system_prompt=WEAK_PROMPT,
        callback_handler=None,
    )
    return str(agent(case.input))

weak_experiment = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator])
weak_report = weak_experiment.run_evaluations(weak_task)
weak_report.run_display(include_actual_output=True)

# Strong agent — has tools, looks up real data
print("\n\u2705 Strong agent (with tools) — expect HIGH scores:")

def strong_task(case: Case) -> str:
    agent = Agent(
        model=get_model(),
        tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff],
        system_prompt=STRONG_PROMPT,
        callback_handler=None,
    )
    return str(agent(case.input))

strong_experiment = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator])
strong_report = strong_experiment.run_evaluations(strong_task)
strong_report.run_display(include_actual_output=True)

print(f"\n\U0001f4ca Weak: {weak_report.overall_score:.2f} vs Strong: {strong_report.overall_score:.2f}")
print("The gap proves the rubric discriminates real knowledge from fabrication.")

## Part 2: Trajectory Eval

Output eval tells you WHAT the agent said. Trajectory eval tells you
HOW it got there. Did it look up the data before claiming facts?

The trajectory evaluator checks that expected tools were called in the
right order. A steered agent that follows lookup-before-claim will
score high. An agent that skips lookups and guesses will score low.

In [ ]:
# --- Trajectory cases ---

trajectory_cases = [
    Case[str, str](
        name="player-stat-workflow",
        input="Tell me about Tom Brady's 2004 season stats.",
        expected_output="Brady's 2004 stats looked up via tools.",
        expected_trajectory=["lookup_player", "get_season_stats"],
    ),
    Case[str, str](
        name="game-lookup",
        input="What happened in the AFC Championship?",
        expected_output="AFC Championship result looked up via tools.",
        expected_trajectory=["get_game_result"],
    ),
    Case[str, str](
        name="position-query",
        input="Who played quarterback for the 2004 Patriots?",
        expected_output="QB roster looked up via tools.",
        expected_trajectory=["get_roster_by_position"],
    ),
]

# --- Build trajectory evaluator ---

trajectory_evaluator = TrajectoryEvaluator(
    rubric="""
    Evaluate whether the agent used the expected tools in the correct order.
    Use the scoring tools provided to verify trajectory matches:
    - The expected tools should appear in order (extra tools between are OK).
    - Score 1.0 if the expected sequence is followed.
    - Score 0.5 if tools are called but in wrong order or incomplete.
    - Score 0.0 if expected tools are missing entirely.
    """,
    include_inputs=True,
)

# Get tool descriptions for the evaluator
sample_agent = Agent(
    tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff]
)
tool_descriptions = tools_use_extractor.extract_tools_description(sample_agent, is_short=True)
trajectory_evaluator.update_trajectory_description(tool_descriptions)

# --- Run trajectory eval ---

def trajectory_task(case: Case) -> dict:
    agent = Agent(
        model=get_model(),
        tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff],
        system_prompt=STRONG_PROMPT,
        callback_handler=None,
    )
    response = agent(case.input)
    trajectory = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
    return {"output": str(response), "trajectory": trajectory}

print("\u2705 Steered agent — expect correct tool sequences:")
trajectory_experiment = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator])
report = trajectory_experiment.run_evaluations(trajectory_task)
report.run_display(include_actual_output=True)
print(f"\n\U0001f4ca Trajectory score: {report.overall_score:.2f}")

## Try It Yourself

Ideas to extend:
- Add a case that tests multi-tool workflows (e.g. roster lookup then game result)
- Tweak the rubric to weight specificity more heavily
- Add a third "medium" agent (tools but no system prompt steering) to see where steering matters
- Create a podcast-delegation trajectory case for module 06's multi-agent pattern

In [ ]:
# Try adding a new output eval case:
#
# Case[str, str](
#     name="win-streak",
#     input="What was the Patriots' win streak in 2004?",
#     expected_output="21 consecutive wins (NFL record), carried from 2003 into 2004, ended Week 8 at Pittsburgh.",
# )
#
# Try adjusting the rubric — what happens if you remove the "narrative connection" criterion?

## Workshop Complete

You've built the full arc:

| Module | Concept | NGS Parallel |
|--------|---------|-------------|
| 01 | Agent loop + tools | 11 features extracted per play |
| 02 | Hooks | CloudWatch monitors every inference |
| 03 | Skills + steering | Trained models carry domain knowledge |
| 04 | Session managers | S3 stores 10+ years of historical data |
| 05 | Deploy | Lambda + API Gateway (event-driven) |
| 06 | Multi-agent | Lean orchestrator delegates to specialists |
| 07 | Evals | 90% directional approval from experts |

Production follow-on: observability (CloudWatch + X-Ray), A/B eval pipelines,
guardrail layers, and the same feedback loop NGS runs between broadcast
approval and model retraining.